# Scene-Centered Metrics · Agent-Type Breakdown

Computes scene-level agent distributions split by type (vehicle, pedestrian, bicycle, motorcycle). Metrics include instantaneous counts, maximum simultaneous agents, and spatial density across observed agents.


## 1. Configure dataset root


In [ ]:
from pathlib import Path
import os

NUSCENES_ROOT = Path('../data/raw').resolve()
if not NUSCENES_ROOT.exists():
    raise FileNotFoundError(f"nuScenes root not found at {NUSCENES_ROOT}. Update the path before continuing.")

os.environ['NUSCENES_ROOT'] = str(NUSCENES_ROOT)
print(f"nuScenes root set to: {NUSCENES_ROOT}")


## 2. Imports and helpers


In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import seaborn as sns
import matplotlib.pyplot as plt

from trajdata import UnifiedDataset
from trajdata.data_structures import AgentType
from trajdata.data_structures import scene_collate_fn

sns.set_style("whitegrid")

AGENT_TYPES = [
    AgentType.VEHICLE,
    AgentType.PEDESTRIAN,
    AgentType.BICYCLE,
    AgentType.MOTORCYCLE,
]
AGENT_TYPE_NAMES = {t.value: t.name.capitalize() for t in AGENT_TYPES}


def agent_type_name(tensor_val) -> str:
    try:
        at = AgentType(int(tensor_val))
        return AGENT_TYPE_NAMES.get(at.value, at.name.capitalize())
    except Exception:
        return str(int(tensor_val))


## 3. Dataset loader (scene-centric)


In [ ]:
def load_scene_dataset(batch_size: int = 12) -> DataLoader:
    dataset = UnifiedDataset(
        desired_data=["nusc_mini"],
        centric="scene",
        desired_dt=0.1,
        history_sec=(3.2, 3.2),
        future_sec=(4.8, 4.8),
        incl_raster_map=False,
        incl_vector_map=False,
        state_format="x,y,xd,yd,xdd,ydd,h",
        obs_format="x,y,xd,yd,xdd,ydd,s,c",
        standardize_data=False,
        standardize_derivatives=False,
        rebuild_cache=False,
        num_workers=0,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=scene_collate_fn,
    )


scene_loader = load_scene_dataset()
first_scene_batch = next(iter(scene_loader))
print("Loaded scene-centric batch with", first_scene_batch.agent_hist.shape[1], "agents")


## 4. Metric helpers


In [ ]:
def append_metric(rows, metric_name: str, agent_types, values):
    values = values.detach().cpu().numpy() if isinstance(values, torch.Tensor) else np.asarray(values)
    for agent_type_val, metric_val in zip(agent_types, values):
        rows.append({
            "metric": metric_name,
            "agent_type": agent_type_name(agent_type_val),
            "value": float(metric_val),
        })


def compute_scene_metrics(loader: DataLoader) -> pd.DataFrame:
    rows = []
    for batch in loader:
        batch_size, num_agents, hist_len, _ = batch.agent_hist.shape
        agent_types = batch.agent_type
        valid_agent_mask = batch.agent_hist_len > 0

        # Instantaneous agent counts per sample and type
        for agent_type in AGENT_TYPES:
            type_mask = (agent_types == int(agent_type)) & valid_agent_mask
            counts = type_mask.sum(dim=1)
            append_metric(rows, "instantaneous_count", torch.full((batch_size,), int(agent_type)), counts)

        # Maximum simultaneous agents per type within the batch
        for agent_type in AGENT_TYPES:
            type_mask = (agent_types == int(agent_type)) & valid_agent_mask
            max_count = type_mask.sum(dim=1).max(dim=-1).values if type_mask.ndim == 3 else type_mask.sum(dim=1)
            append_metric(rows, "max_simultaneous_agents", torch.full((batch_size,), int(agent_type)), max_count)

        # Spatial density: count / area of bounding box of valid agent positions
        positions = batch.agent_hist.position  # (B, N, T, 2)
        time_mask = torch.arange(hist_len, device=positions.device).unsqueeze(0).unsqueeze(0) < batch.agent_hist_len.unsqueeze(-1)
        valid_positions = torch.where(time_mask.unsqueeze(-1), positions, torch.full_like(positions, float("nan")))
        min_xy = torch.nanmin(valid_positions, dim=(1, 2)).values
        max_xy = torch.nanmax(valid_positions, dim=(1, 2)).values
        span = (max_xy - min_xy).clamp(min=1e-3)
        area = span[..., 0] * span[..., 1]
        total_counts = valid_agent_mask.sum(dim=1).float()
        overall_density = total_counts / area
        append_metric(rows, "spatial_density_agents_per_m2", torch.full((batch_size,), AgentType.VEHICLE.value), overall_density)

        # Type-specific density based on counts only (area reused)
        for agent_type in AGENT_TYPES:
            type_mask = (agent_types == int(agent_type)) & valid_agent_mask
            type_counts = type_mask.sum(dim=1).float()
            density = type_counts / area
            append_metric(rows, "spatial_density_agents_per_m2_type", torch.full((batch_size,), int(agent_type)), density)

    return pd.DataFrame(rows)


## 5. Run scene metrics and export


In [ ]:
scene_metrics_df = compute_scene_metrics(scene_loader)
scene_summary = (
    scene_metrics_df
    .groupby(["metric", "agent_type"])["value"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .reset_index()
)

metrics_dir = Path("../results/metrics")
metrics_dir.mkdir(parents=True, exist_ok=True)
scene_metrics_df.to_csv(metrics_dir / "scene_centered_metrics_by_type.csv", index=False)
scene_summary.to_csv(metrics_dir / "scene_centered_metrics_by_type_summary.csv", index=False)
scene_summary.head()


## 6. Visualize agent-type comparisons


In [ ]:
order = ["Vehicle", "Pedestrian", "Bicycle", "Motorcycle"]

plt.figure(figsize=(8, 4))
subset = scene_metrics_df[scene_metrics_df.metric == "instantaneous_count"]
sns.boxplot(data=subset, x="agent_type", y="value", order=order)
plt.title("Instantaneous agent counts by type")
plt.ylabel("Count")
plt.show()

density_subset = scene_metrics_df[scene_metrics_df.metric == "spatial_density_agents_per_m2_type"]
sns.catplot(data=density_subset, x="agent_type", y="value", order=order, kind="bar")
plt.title("Spatial density by agent type")
plt.tight_layout()
plt.show()
